# Selection Efficiency audit (Colab)

**Runtime → Change runtime type → T4 GPU.** Then run all cells.

Everything is written to your Google Drive (`MyDrive/se_audit_run`) after every batch, so **if Colab disconnects, nothing is lost**: reconnect, choose Runtime → Run all again, and it continues from the last checkpoint. Finished splits are skipped without reloading the model.

Run only one Colab session at a time on this folder. A second session is refused while the first is alive.

**Math splits changed.** MA-1/MA-2 (MultiArith) are replaced by GSM-3 and GSM-5 (GSM8K problems with exactly 3, or 5 or more, calculation steps). If you already have the Web-of-Lies results from the earlier run, put that run's `raw_outputs.zip` in `MyDrive/se_audit_run` and **rename it `earlier_raw_outputs.zip`** (the new run writes its own `raw_outputs.zip` there). The next cell unpacks it once, and only the GSM splits are generated: about 1.5–2.5 h on one T4. Without it, everything is regenerated (about 3 h).

In [ ]:
REPO = 'https://github.com/manyagupta13/se-audit.git'
WORK = '/content/drive/MyDrive/se_audit_run'   # checkpoints live here
MODE = 'full'                                   # 'full', 'smoke', or 'status'

In [ ]:
# Mount Drive and get the latest code
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.isdir('/content/se_audit/.git'):
    !git -C /content/se_audit pull -q
else:
    !git clone -q {REPO} /content/se_audit
!git -C /content/se_audit log -1 --format='code version: %h  %s'
!pip install -q 'sentence-transformers>=2.6' scipy 2>&1 | tail -1
!nvidia-smi -L

In [ ]:
# One time: reuse the finished Web-of-Lies results from the earlier run (skips if nothing to reuse)
import zipfile, os
OUT = os.path.join(WORK, 'se_audit_out')
Z = os.path.join(WORK, 'earlier_raw_outputs.zip')
if os.path.isdir(os.path.join(OUT, 'gen')):
    print('Already have outputs in', OUT, '- nothing to unpack.')
elif os.path.exists(Z):
    n = 0
    with zipfile.ZipFile(Z) as z:
        for m in z.namelist():
            base = os.path.basename(m).split('.')[0]
            if m.endswith('/') or base in ('ma1', 'ma2'):      # old MultiArith outputs are not used any more
                continue
            dest = os.path.realpath(os.path.join(OUT, m))
            if not dest.startswith(os.path.realpath(OUT) + os.sep):
                continue
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            with z.open(m) as src, open(dest, 'wb') as dst:
                dst.write(src.read())
            n += 1
    print('Unpacked', n, 'files from', Z)
else:
    print('No earlier_raw_outputs.zip in', WORK, '- everything will be generated from scratch.')


In [ ]:
# Where things stand (safe to run any time)
!python /content/se_audit/run_pipeline.py --work {WORK} --status

In [ ]:
# Run (resumes automatically). Progress is printed every 5 minutes.
!python /content/se_audit/run_pipeline.py --work {WORK} --{MODE}

In [ ]:
# Download the results (also kept in Drive at WORK)
from google.colab import files
for f in ['tables.zip', 'raw_outputs.zip']:
    p = os.path.join(WORK, f)
    print(p, 'exists' if os.path.exists(p) else 'not yet')
    if os.path.exists(p): files.download(p)

**If the run cell is refused with “Another session … may still be running”:** the previous runtime wrote to the folder in the last 2–3 minutes. Wait that long and rerun, or, if you are sure the old session is gone, add `--force` to the run command.